# Geometry Working Memory

https://www.biorxiv.org/content/10.64898/2026.08.31.748237v1

In [ ]:
# libraries

import os
import numpy as np
import pickle
import pandas as pd
from scipy.linalg import orthogonal_procrustes


In [ ]:
### set paths and settings

path_root = '/path_to_local'

# settings

subjects = [f'sub_{i:02d}' for i in range(1, 50)]

time_windows = ['encode', 'maint', 's2']
pca_aligned_folder = 'pca_aligned'
k_start, k_end = 0, 2       # 0,2 means leading three PCs

outputs = [
'control_2gratings_2polygons',                                  'control_1gratings_1polygons',
'update_2gratings_relevant_2polygons_nonrelevant',              'update_1gratings_relevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_relevant',              'update_1gratings_nonrelevant_1polygons_relevant',
'inhibition_2gratings_relevant_2polygons_nonrelevant',          'inhibition_1gratings_relevant_1polygons_nonrelevant',
'inhibition_2gratings_nonrelevant_2polygons_relevant',          'inhibition_1gratings_nonrelevant_1polygons_relevant',
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      'update_1gratings_nolongerrelevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',      'update_1gratings_nonrelevant_1polygons_nolongerrelevant',
]


# Compute procrustes distances

In [10]:
# loop over subjects
for sub_idx, sub_i in enumerate(subjects):

    path_outputs = os.path.join(path_root, 'results', pca_aligned_folder, 'procrustes_distance_subspaces' + str(k_start) + 'to' + str(k_end), sub_i)
    if not os.path.isdir(path_outputs):
        os.makedirs(path_outputs)

    if not os.path.isfile(os.path.join(path_outputs, 'procrustes_shape' + '_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl')):

        print(sub_i)

        procrustes_joint = {}
        procrustes_orientation = {}
        procrustes_shape = {}

        # loop over conditions (including stimulus load)
        for output_idx, output_i in enumerate(outputs):

            pc_scores = []

            # loop over locked time segments
            for time_window in time_windows:

                # load file with pca outputs
                file = os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', sub_i, 'pca_correct_trials.pkl')
                with open(file, 'rb') as file:
                    data = pickle.load(file)

                if time_window == 'encode':

                    segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

                elif time_window == 'maint':

                    segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

                elif time_window == 's2':

                    segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

                # loop over timepoints
                for time_idx, time_segment in enumerate(segments):

                    key = time_window + '_segment' + str(time_segment) + '_' + output_i

                    pc_scores.append(data['pc_scores_' + key][:,k_start:(k_end+1)])

            ## procrustes (no scale and translate) - joint PC scores
            procrustes_distances = np.full((len(pc_scores), len(pc_scores)), np.nan)

            for time_i in range(len(pc_scores)):
                for time_ii in range(len(pc_scores)):

                    # 1. Center the matrices to remove translation
                    X = pc_scores[time_i] - np.mean(pc_scores[time_i], axis=0, keepdims=True)
                    Y = pc_scores[time_ii] - np.mean(pc_scores[time_ii], axis=0, keepdims=True)

                    # 2. Compute optimal rotation
                    R, scale = orthogonal_procrustes(X, Y)

                    # 3. Apply rotation (without scaling)
                    X_aligned = X @ R
                    # With scaling, uncomment the next line:
                    # X_aligned = X @ R * scale

                    # 4. Compute Procrustes distance (Frobenius norm)
                    procrustes_distances[time_i, time_ii] = np.linalg.norm(Y - X_aligned, 'fro')

            procrustes_joint[output_i] = procrustes_distances


            ## procrustes (no scale and translate) - orientation PC scores
            procrustes_distances = np.full((len(pc_scores), len(pc_scores)), np.nan)

            for time_i in range(len(pc_scores)):
                for time_ii in range(len(pc_scores)):

                    # 1. Center the matrices to remove translation
                    X = pc_scores[time_i][0:4,:] - np.mean(pc_scores[time_i][0:4,:], axis=0, keepdims=True)
                    Y = pc_scores[time_ii][0:4,:] - np.mean(pc_scores[time_ii][0:4,:], axis=0, keepdims=True)

                    # 2. Compute optimal rotation 
                    R, scale = orthogonal_procrustes(X, Y)

                    # 3. Apply rotation (ithout scaling)
                    X_aligned = X @ R
                    # With scaling, uncomment the next line:
                    # X_aligned = X @ R * scale

                    # 4. Compute Procrustes distance (Frobenius norm)
                    procrustes_distances[time_i, time_ii] = np.linalg.norm(Y - X_aligned, 'fro')

            procrustes_orientation[output_i] = procrustes_distances

            ## procrustes (no scale and translate) - shape PC scores
            procrustes_distances = np.full((len(pc_scores), len(pc_scores)), np.nan)

            for time_i in range(len(pc_scores)):
                for time_ii in range(len(pc_scores)):

                    # 1. Center the matrices to remove translation
                    X = pc_scores[time_i][4:7,:] - np.mean(pc_scores[time_i][4:7,:], axis=0, keepdims=True)
                    Y = pc_scores[time_ii][4:7,:] - np.mean(pc_scores[time_ii][4:7,:], axis=0, keepdims=True)

                    # 2. Compute optimal rotation (and scaling if desired)
                    R, scale = orthogonal_procrustes(X, Y)

                    # 3. Apply rotation (ithout scaling)
                    X_aligned = X @ R
                    # With scaling, uncomment the next line:
                    # X_aligned = X @ R * scale

                    # 4. Compute Procrustes distance (Frobenius norm)
                    procrustes_distances[time_i, time_ii] = np.linalg.norm(Y - X_aligned, 'fro')

            procrustes_shape[output_i] = procrustes_distances


        with open(os.path.join(path_outputs, 'procrustes_joint' + '_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
            pickle.dump(procrustes_joint, file)

        with open(os.path.join(path_outputs, 'procrustes_orientation' + '_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
            pickle.dump(procrustes_orientation, file)

        with open(os.path.join(path_outputs, 'procrustes_shape' + '_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
            pickle.dump(procrustes_shape, file)


sub_01
sub_02
sub_03
sub_04
sub_05
sub_06
sub_07
sub_08
sub_09
sub_10
sub_11
sub_12
sub_13
sub_14
sub_15
sub_16
sub_17
sub_18
sub_19
sub_20
sub_21
sub_22
sub_23
sub_24
sub_25
sub_26
sub_27
sub_28
sub_29
sub_30
sub_31
sub_32
sub_33
sub_34
sub_35
sub_36
sub_37
sub_38
sub_39
sub_40
sub_41
sub_42
sub_43
sub_44
sub_45
sub_46
sub_47
sub_48
sub_49
